# 05: U-Net na zbiorze z nagrań (RAFT + geometria + U-Net)

**Punkt wyjścia (04):** gotowy RAFT + geometria daje 32,86 dB, dostrojony RAFT **34,30 dB** (wejście 26,42 dB),
z nasyceniem od ~5. epoki. Dalsze dostrajanie samego RAFT niewiele da, bo geometria ma ograniczenia, których żaden przepływ nie naprawi:
* piksele, których **A w ogóle nie widzi** (wyszły poza kadr, tło odsłonięte przez poruszający się obiekt): `warp(A)` nie ma ich skąd wziąć,
* pas wzdłuż antyprzekątnej, gdzie obie kamery czytają punkt w tej samej chwili: prędkość jest tylko uzupełniana z sąsiedztwa,
* ruch niejednostajny w czasie odczytu (obrót, przyspieszenie), rozmycie interpolacji, różnice jasności i szum.

**Podejście:** U-Net nie zastępuje geometrii, tylko ją **doszlifowuje** (jak etap 2 dla MotionNet w 03):
1. RAFT (wagi z 04, zamrożony) + geometria: przepływ A→B, prędkość każdego piksela, obraz A w chwili docelowej.
2. Tą samą prędkością przenosimy do chwili docelowej również **obraz B**.
3. U-Net dostaje: A po korekcji, B po korekcji, maski „piksel w kadrze A / B”, pewność geometrii, prędkość, współrzędne.
   Zwraca: lokalne poprawki przepływu A i B, **maskę wyboru kamery** i drobną poprawkę obrazu.
   `wynik = maska · warp(A) + (1 − maska) · warp(B) + poprawka`
4. Strata: L1 obrazu tam, gdzie scenę widziała choć jedna kamera.

**Porównanie (do pracy):** zwykły U-Net bez RAFT i bez geometrii (wejście: A, B, współrzędne, czasy odczytu pikseli).
Na zbiorze z ruchem kamery analogiczny `flow_dual` prawie się nie uczył (20,45 dB przy wejściu 19,66 dB), więc spodziewamy się słabego wyniku;
to argument za geometrią.

| sekcja | co | czas na L4 (szacunek) |
|---|---|---|
| 3 | zwykły U-Net, 30 epok | ~15 min |
| 4 | RAFT (zamrożony) + U-Net, 30 epok | ~35 min |
| 5 | opcjonalnie: wspólne dostrajanie RAFT + U-Net, 10 epok | ~20 min |
| 6–7 | tabela wyników, podział na siłę RS, podglądy | ~5 min |

Kod: `src/raft_unet.py` (modele), `src/train_unet.py` (trening i ewaluacja). **Przed uruchomieniem zrób commit + push**, bo SETUP pobiera kod z GitHuba.

In [ ]:
# === SETUP: uruchamiaj zawsze jako pierwszą komórkę ===
# Kod jest pobierany z GitHuba na maszynę Colaba (lokalne pliki z VS Code tam nie trafiają).
# Po zmianie kodu: commit + push w VS Code, potem ponownie ta komórka (robi git pull).
REPO = 'kordiusz/RollingShutter'   # repozytorium na GitHubie
BRANCH = 'main'

import os, sys, subprocess, getpass, importlib
from google.colab import drive
drive.mount('/content/drive')

TOKEN_FILE = '/content/drive/MyDrive/rs_project/.github_token'   # opcjonalnie: token zapisany na Drive
CODE = '/content/code'

def _git(*args):
    r = subprocess.run(['git', *args], capture_output=True, text=True)
    if r.returncode != 0:
        msg = (r.stderr or r.stdout).replace(token, '***') if 'token' in globals() else r.stderr
        raise RuntimeError('git ' + args[0] + ' nie powiodło się:\n' + msg)
    return r.stdout

if not os.path.exists(CODE):
    token = (open(TOKEN_FILE).read().strip() if os.path.exists(TOKEN_FILE)
             else getpass.getpass('GitHub token (fine-grained, tylko odczyt): '))
    _git('clone', '-q', '-b', BRANCH, f'https://{token}@github.com/{REPO}.git', CODE)
else:
    _git('-C', CODE, 'pull', '-q')
os.chdir(CODE)
sys.path.insert(0, f'{CODE}/src')
import project_paths as P
importlib.reload(P)
P.ensure_dirs()
print(_git('log', '-1', '--format=kod: %h  %s  (%cd)', '--date=short'))
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 1. Zbiór danych (rozpakowanie z Drive na dysk maszyny Colaba)
Ten sam zbiór co w 04. Po wygenerowaniu wersji 2 wystarczy zmienić `DATASET_NAME`.

In [ ]:
import os, glob
DATASET_NAME = 'rs_video_x4k'
z = P.find_dataset_zip(DATASET_NAME)
assert z, f'Brak {DATASET_NAME}.zip w {P.DATASETS} - wrzuć tam spakowany folder data/{DATASET_NAME}'
UNZ = f'/content/unz_{DATASET_NAME}'
if not os.path.exists(UNZ):
    print('rozpakowuję', z, '...')
    !unzip -q "{z}" -d "{UNZ}"
# folder z podfolderami train/ i val/ (niezależnie od tego, jak spakowano)
DATA = next(os.path.dirname(p) for p in sorted(glob.glob(f'{UNZ}/**/train', recursive=True)))
print('DATA =', DATA, '|', len(os.listdir(f'{DATA}/train')), 'train /', len(os.listdir(f'{DATA}/val')), 'val')
WORKERS = min(8, os.cpu_count())

## 2. RAFT z 04
Dostrojone wagi RAFT z `runs/<data>_raft_geom/best.pt`. Bez nich U-Net startuje z gotowego RAFT (wynik 04 bez treningu).

In [ ]:
from datetime import date
from IPython.display import Image, display
RAFT_CKPT = P.find_run('raft_geom')
print('RAFT:', RAFT_CKPT or 'brak dostrojonego - użyję gotowych wag RAFT')
raft_arg = f'--raft-ckpt "{RAFT_CKPT}"' if RAFT_CKPT else ''
# nazwy runów; przy wznawianiu innego dnia wpisz tu nazwy istniejących folderów z runs/
RUN_PLAIN = f'{date.today()}_unet_plain'
RUN_REF = f'{date.today()}_raft_unet'
RUN_FT = f'{date.today()}_raft_unet_ft'

## 3. Porównanie: zwykły U-Net (bez RAFT i geometrii)
Po rozłączeniu: SETUP, rozpakowanie i sekcja 2 (nazwy runów), potem w danej komórce `RESUME = True`.

In [ ]:
RESUME = False
resume = f'--resume "{P.RUNS / RUN_PLAIN / "last.pt"}"' if RESUME else ''
!python src/train_unet.py --data "{DATA}" --mode plain --epochs 30 --bs 16 --lr 3e-4 \
    --workers {WORKERS} --out "{P.RUNS}" --name {RUN_PLAIN} {resume}

## 4. Model główny: RAFT (zamrożony) + geometria + U-Net
Przed treningiem wynik ≈ RAFT + geometria (U-Net startuje prawie od zera). W logu: `RAFT+geom` = sam etap 1 na tych samych próbkach,
`from B` = jaki ułamek obrazu U-Net bierze z kamery B.

In [ ]:
RESUME = False
resume = f'--resume "{P.RUNS / RUN_REF / "last.pt"}"' if RESUME else ''
!python src/train_unet.py --data "{DATA}" --mode refine {raft_arg} --epochs 30 --bs 8 --lr 3e-4 \
    --workers {WORKERS} --out "{P.RUNS}" --name {RUN_REF} {resume}

## 5. (opcjonalnie) Wspólne dostrajanie RAFT + U-Net
Start z wag z sekcji 4; RAFT uczy się z małym krokiem (`--raft-lr 1e-5`), U-Net z mniejszym niż wcześniej.
Ma sens, jeśli w sekcji 4 wynik rósł do końca. Pomiń, jeśli brakuje czasu na GPU.

In [ ]:
!python src/train_unet.py --data "{DATA}" --mode refine --init "{P.RUNS / RUN_REF / 'best.pt'}" \
    --raft-lr 1e-5 --lr 1e-4 --epochs 10 --bs 8 --workers {WORKERS} --out "{P.RUNS}" --name {RUN_FT}

## 6. Wyniki na val
* `psnr`, `ssim`: obszar, w którym piksel źródłowy jest w kadrze A (tak samo jak w 04, więc wiersze RAFT są porównywalne),
* `psnr_full`: cały obraz (tu liczą się też brzegi, które może uzupełnić tylko kamera B),
* `*_in`: bez korekcji, `psnr_geom`: sam RAFT + geometria w tym samym modelu,
* `ms_per_pair`: czysty czas sieci dla jednej pary (bez wczytywania danych).

In [ ]:
import json, pandas as pd
rows, per = [], {}
for name in ['raft_zeroshot', 'raft_geom']:                      # wyniki z 04 (eval.json zapisany w 04)
    ej = P.find_run(name, 'eval.json')
    if ej:
        rows.append({'run': ej.parent.name, **json.load(open(ej))['metrics']})
for run in [RUN_PLAIN, RUN_REF, RUN_FT]:
    ck = P.RUNS / run / 'best.pt'
    if not ck.exists():
        print('brak', ck); continue
    ej, ps = P.RUNS / run / 'eval.json', P.RUNS / run / 'val_per_sample.csv'
    !python src/train_unet.py --data "{DATA}" --eval "{ck}" --eval-json "{ej}" --per-sample "{ps}" --bs 8 --workers {WORKERS} > /dev/null
    rows.append({'run': run, **json.load(open(ej))['metrics']})
    per[run] = pd.read_csv(ps)
cols = ['psnr', 'ssim', 'psnr_full', 'psnr_geom', 'psnr_in', 'ssim_in', 'valid', 'blend_b', 'ms_per_pair', 'params_M']
tab = pd.DataFrame(rows).set_index('run')
display(tab[[c for c in cols if c in tab]].round(3))

## 7. Wyniki w podziale na siłę RS
Średnia z całego val jest zaniżona przez próbki prawie bez ruchu. Klasy z `src/analyze_dataset.py`
(95. percentyl przesunięcia RS: słaby < 2 px ≤ średni < 6 px ≤ silny). Liczone tylko dla val.

In [ ]:
VAL_ONLY, ANA = '/content/val_only', '/content/analiza_val'
if not os.path.exists(f'{ANA}/samples.csv'):
    os.makedirs(VAL_ONLY, exist_ok=True)
    if not os.path.exists(f'{VAL_ONLY}/val'):
        os.symlink(f'{DATA}/val', f'{VAL_ONLY}/val')
    !python src/analyze_dataset.py --data {VAL_ONLY} --out {ANA} --workers {WORKERS} > /dev/null
cls = pd.read_csv(f'{ANA}/samples.csv').query('split == "val"')[['sample', 'class', 'disp_p95']]
t = cls.copy()
for run, df in per.items():
    df = df.set_index('sample')
    if 'psnr_in' not in t:
        t = t.join(df[['psnr_in']], on='sample')
    if 'psnr_geom' in df and 'RAFT+geom' not in t:
        t = t.join(df['psnr_geom'].rename('RAFT+geom'), on='sample')
    t = t.join(df['psnr'].rename(run), on='sample')
order = ['slaby', 'sredni', 'silny']
summary = t.drop(columns=['sample']).groupby('class').agg(['mean']).droplevel(1, axis=1).reindex(order)
summary.insert(0, 'n', t.groupby('class').size().reindex(order).fillna(0).astype(int))
display(summary.round(2))
print('PSNR [dB] w obszarze widocznym w A; disp_p95 = średnia siła RS w klasie [px]')

In [ ]:
for run in [RUN_PLAIN, RUN_REF, RUN_FT]:
    ps = sorted((P.RUNS / run).glob('preview_ep*.png'))
    if ps:
        print(ps[-1]); display(Image(str(ps[-1])))
print('refine: RS A | RS B | RAFT+geom | jego błąd | U-Net | jego błąd | wzorzec | maska (biała = A, czarna = B)')
print('plain:  RS A | RS B | U-Net | jego błąd | wzorzec')

## 8. Po treningu
Dopisz wiersze do `experiments/README.md` (`*_unet_plain`, `*_raft_unet`, ew. `*_raft_unet_ft`: PSNR/SSIM z tabeli w sekcji 6,
`ms_per_pair`, parametry) i tabelę z sekcji 7 do wniosków. Commit notatnika razem z wynikami.